<a href="https://colab.research.google.com/github/priyalimbu246/Valson-course/blob/main/Copy_of_Lecture_13_Regression_with_LightGBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Lecture 13 - Regression

Here we are going to regression modeling on the Bradley Melting Point Dataset, which is curated chemical dataset with melting points of around 3,000 chemical compounds, see [here](https://www.kaggle.com/datasets/aliffaagnur/melting-point-chemical-dataset/data).

We will use the [LightGBM](https://en.wikipedia.org/wiki/LightGBM) method is seperate from scikit-learn but uses the same API.

This is heavily inspired by this tutorial:
- [Building a Simple Regression Model](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/ml_models/regression_model.ipynb)


Install RDKit and [LightGBM](https://lightgbm.readthedocs.io/en/stable/)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit lightgbm mols2grid

Import all basic pacakges

In [ ]:
# basic
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# For progress bar
from tqdm.auto import tqdm

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

# LightGBM
from lightgbm import LGBMRegressor, plot_importance

# mols2grid
import mols2grid

# To visluzise results
from yellowbrick.regressor import prediction_error, ResidualsPlot

Download dataset

In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/valsson-group/UNT-ChemicalApplicationsOfMachineLearning-Spring2026/refs/heads/main/Assignment-2/"
dataset_filename="BradleyDoublePlusGoodMeltingPointDataset.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

This is to add progress bars panda task

In [ ]:
tqdm.pandas()

Read dataset

In [ ]:
data_mp = pd.read_csv("BradleyDoublePlusGoodMeltingPointDataset.csv")



In [ ]:
data_mp

In [ ]:
# Simplify by removing columns from the data frame
data_mp = data_mp.drop(columns=['csid','link','source','count','min','max','range'])

Let's visualize the molecules using mols2grid.

In [ ]:
def mp_str(x):
    return f'{x:.2f} C'

mols2grid.display(data_mp,
                  smiles_col='smiles',
                  subset=['img','name','mpC'],
                  transform={"mpC": mp_str})

## SMILES ARbitrary Target Specification (SMARTS)

SMARTS is a very useful method to look for substructures in molecules using RDKit and it can be used in various ways in RDKit, for example to delete parts. We can use it to search in the mols2grid above.

See these excellent tutorials for further information about SMARTS:
- [An introduction to the SMILES ARbitrary Target Specification (SMARTS)](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/fundamentals/SMARTS_tutorial.ipynb)
- [Recursive SMARTS](https://colab.research.google.com/github/PatWalters/practical_cheminformatics_tutorials/blob/main/fundamentals/recursive_smarts.ipynb)
- (SMARTS Tutorial)[https://www.daylight.com/dayhtml_tutorials/languages/smarts/]

For example, we can use the following:
- Benzene rings: `c1ccccc1[#6]`
- Atoms with charges -1 or +1: `[-1]` or `[+1]`
- Nitrogen or oxygen attached to an aliphatic carbon: `C[#7,#8]`

Try to search for certain SMARTS string in the mols2grid object above.

## Calculate RDKit Features

Here we use a quick way to calculate a large set of properties. Will be explained in class.

In [ ]:
property_names = list(rdMolDescriptors.Properties.GetAvailableProperties())


In [ ]:
property_names

In [ ]:
property_getter = rdMolDescriptors.Properties(property_names)

In [ ]:
property_getter

In [ ]:
mol=Chem.MolFromSmiles("c1ccccc1CC")
print(np.array(property_getter.ComputeProperties(mol)))



In [ ]:
property_names_2=['tpsa','NumRotatableBonds']
property_getter_2 = rdMolDescriptors.Properties(property_names_2)
mol=Chem.MolFromSmiles("c1ccccc1CC")
print(np.array(property_getter_2.ComputeProperties(mol)))

In [ ]:
def smi2props(smi):
    mol = Chem.MolFromSmiles(smi)
    props = None
    if mol:
        Chem.DeleteSubstructs(mol, Chem.MolFromSmarts("[#1X0]"))
        props = np.array(property_getter.ComputeProperties(mol))
    return props

In [ ]:
data_mp['props']=[smi2props(s) for s in data_mp['smiles']]

In [ ]:
# These two ways are equilivant

data_mp['props'] = data_mp.smiles.progress_apply(smi2props)

# data_mp['props'] = data_mp['smiles'].progress_apply(smi2props)

In [ ]:
# data_mp = data_mp.dropna(subset=['props']) # The 'props' column was already processed and dropped.

In [ ]:
data_mp

In [ ]:
# data_mp[property_names] = data_mp['props'].tolist() # This line caused an error because the 'props' column was already dropped.

In [ ]:
# data_mp.drop("props",axis=1,inplace=True) # This column was already dropped in a previous execution

## Train using LightGBM

In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)

In [ ]:
train_X = train[property_names]
train_y = train.mpC
test_X = test[property_names]
test_y = test.mpC

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)

In [ ]:
pred = lgbm.predict(test_X)

In [ ]:
ax = sns.scatterplot(x=test_y,y=pred)
ax.set(xlabel="Experimental mpC")
ax.set(ylabel="Predicted mpC")

In [ ]:
ax = sns.regplot(x=test_y,y=pred,scatter_kws={'s':10})
ax.set(xlabel="Experimental mpC")
ax.set(ylabel="Predicted mpC")

In [ ]:
r2_score(test_y,pred)

In [ ]:
mean_squared_error(test_y,pred)

[Yellowbrick](https://www.scikit-yb.org/en/latest/) is python package that extends scikit-learn in various ways, including visualization, that we use here.

In [ ]:


visualizer = prediction_error(lgbm, train_X, train_y, test_X, test_y,alpha=0.35)

In [ ]:
visualizer = ResidualsPlot(lgbm)
visualizer.fit(train_X, train_y)
visualizer.score(test_X, test_y)
visualizer.show();

### Feature Importance

Let's do feature importance

In [ ]:
features_importance = lgbm.feature_importances_

##log10 Transform

In [ ]:
data_mp['mpC_log10'] = np.log10(data_mp['mpC'])

In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)
train_X = train[property_names]
train_y = train.mpC_log10
test_X = test[property_names]
test_y = test.mpC_log10

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)
test_pred = lgbm.predict(test_X)
test_pred_exp = 10**test_pred

In [ ]:
plt.plot(test_y,test_pred,'.',label='Test set')
plt.plot(test_y,test_pred_exp,'.',label='Test set')
plt.xlabel('Experimental mpC')
plt.ylabel('Predicted mpC')
plt.legend()

##Scramble

In [ ]:
train, test = train_test_split(data_mp,test_size=0.20)
train_X = train[property_names]
train_y = train.mpC_log10
test_X = test[property_names]
test_y = test.mpC_log10

In [ ]:
lgbm = LGBMRegressor()
lgbm.fit(train_X, train_y)
test_pred = lgbm.predict(test_X)
test_pred_exp = 10**test_pred

## Cross Validation

In [ ]:
r2_list = []
rmse_list = [] # Initialize rmse_list
NumberOfRandomShuffles=100
for i in tqdm(range(0,NumberOfRandomShuffles)):
    # setup training and test sets
    train, test = train_test_split(data_mp)
    train_X = train[property_names]
    train_y = train.mpC
    test_X = test[property_names]
    test_y = test.mpC
    # create the regressor
    lgbm = LGBMRegressor()
    # train the model
    lgbm.fit(train_X,train_y)
    pred = lgbm.predict(test_X)
    r2 = r2_score(test_y,pred)
    rmse = np.sqrt(mean_squared_error(test_y,pred)) # Calculate RMSE by taking sqrt of MSE
    print(f"R^2: {r2:.3f}, RMSE: {rmse:.3f}") # Print both metrics for feedback
    r2_list.append(r2)
    rmse_list.append(rmse) # Append RMSE

In [ ]:
r2_mean=np.mean(r2_list)
r2_stddeve=np.std(r2_list)
rmse_mean=np.mean(rmse_list)
rmse_stddeve=np.std(rmse_list)
print(f"R^2: {r2_mean:.3f} +- {r2_stddeve:.3f}")
print(f"RMSE: {rmse_mean:.3f} +- {rmse_stddeve:.3f}")

In [ ]:
ax = sns.boxplot(x=r2_list)
ax.set(xlim=(0,1))
ax.set(xlabel="R$^2$")

### Random Forrest

Repeat the analysis using Random Forrest Regreesion

##Home Work